# OmniMind Paper A — Reprodução dos Experimentos MPS

Este notebook carrega `omnimind_paper_a_mps_reproducao.sqlite` e mostra como as seções do artigo podem ser montadas a partir do banco. O banco foi gerado a partir de `mps_bridge_unified_results.json` e `multi_model_dodecatiad_comparison.json` (ambos sanitizados).

## 1. Conexão e panorama do banco

In [ ]:
import sqlite3, pandas as pd, json
DB = 'omnimind_paper_a_mps_reproducao.sqlite'
conn = sqlite3.connect(DB)

def tables(conn):
    return pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

tables(conn)

## 2. Modelos incluídos (Seção 4 do paper)

In [ ]:
pd.read_sql('SELECT * FROM mps_models', conn)

In [ ]:
pd.read_sql('SELECT model_id, params_M, hidden_size, n_layers FROM dodecatiad_models', conn)

## 3. Métricas MPS e χ=4 (Seções 5.2 e 5.3)

In [ ]:
pd.read_sql('''
SELECT model_hf_id, COUNT(*) AS n_conv, AVG(delta_chi4) AS mean_delta_chi4,
       AVG(n_turns) AS mean_n_turns
FROM mps_conversations
GROUP BY model_hf_id
''', conn)

In [ ]:
pd.read_sql('''
SELECT model_hf_id, turn_idx, AVG(mid_chi4) AS mean_chi4, AVG(mid_entropy) AS mean_entropy
FROM mps_turns
GROUP BY model_hf_id, turn_idx
ORDER BY model_hf_id, turn_idx
''', conn)

## 4. Dodecatíade por casa e camada (Seções 5.5 e 5.6)

In [ ]:
pd.read_sql('''
SELECT dm.model_id, dh.house, AVG(dh.energy) AS mean_energy,
       AVG(dh.eff_rank) AS mean_eff_rank, AVG(dh.entropy) AS mean_entropy
FROM dodecatiad_houses dh
JOIN dodecatiad_layers dl ON dh.layer_id = dl.id
JOIN dodecatiad_prompts dp ON dl.prompt_id = dp.id
JOIN dodecatiad_models dm ON dp.model_id = dm.model_id
GROUP BY dm.model_id, dh.house
ORDER BY dm.model_id, dh.house
''', conn)

## 5. Conversas por categoria (Seções 5.7 a 5.10)

In [ ]:
pd.read_sql('''
SELECT category, COUNT(*) AS n_conv, AVG(delta_chi4) AS mean_delta_chi4,
       AVG(accuracy) AS mean_accuracy
FROM mps_conversations
GROUP BY category
''', conn)

## 6. Checklist de reprodução

Para replicar os números do artigo:
- Verifique que as tabelas acima correspondem às tabelas do paper.
- Compare `mid_chi4` com o piso χ=4 declarado.
- Verifique que as casas D12/D13/D15/D27 são computadas por *engine*, não por fatias do *hidden state*.
- Para reexecução completa, os notebooks Kaggle/Colab geraram os JSONs-fonte.

In [ ]:
conn.close()